# 02. Gemini로 튜토리얼 단계 초안 만들기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Reasonofmoon/hangeorum-playground/blob/main/notebooks/02_gemini_step_drafts.ipynb)

> **중요: AI 초안은 그대로 게시하지 않습니다.**
> 사람이 한 줄씩 읽고, 메뉴 이름과 명령을 공식 문서와 대조한 뒤에만 `steps` 탭에 넣습니다.
> AI는 없는 버튼이나 틀린 명령을 그럴듯하게 만들 수 있습니다.

**준비물**
1. Google AI Studio(https://aistudio.google.com/api-keys)에서 **Create API key**로 만든 키
2. 이 노트북 왼쪽 패널 **Secrets**(열쇠 아이콘)에 이름 `GOOGLE_API_KEY`로 저장한 키 (이 노트북의 접근 허용 켜기)

**만드는 것**: `steps` 탭에 붙여 넣을 CSV
열: `tutorial_id, order, code, title, instruction, expected, recovery, kind, sourceType`

**모델**: `gemini-3.8-flash`
출처: https://ai.google.dev/gemini-api/docs/models (2026-10 확인. 새 프로젝트에는 3.5 Flash-Lite 또는 3.8 Flash 사용 권장). 모델이 바뀌면 아래 `MODEL` 칸만 고치세요.

In [ ]:
# 1) SDK 설치 (Google Gen AI SDK, 패키지 이름: google-genai)
# SDK README는 다음 주 버전 변경에 대비해 3.0.0 미만으로 고정하라고 안내합니다.
!pip install -q -U "google-genai<3.0.0"

In [ ]:
#@title 2) 클라이언트 만들기
MODEL = "gemini-3.8-flash" #@param {type:"string"}

from google import genai
from google.genai import types
from google.colab import userdata

# 키는 Colab Secrets에서만 읽습니다. 코드나 출력에 키를 적지 마세요.
client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))
print("클라이언트 준비 완료 / 모델:", MODEL)

In [ ]:
#@title 3) 주제 입력
TOPIC = "구글 시트에서 필터 만들기" #@param {type:"string"}
TUTORIAL_ID = "google-workspace/gw-99" #@param {type:"string"}
CODE_PREFIX = "GW99" #@param {type:"string"}
STEP_COUNT = 6 #@param {type:"slider", min:3, max:8, step:1}

## 4) 응답 형식(JSON 스키마)
사이트의 단계 형식 중 사람이 써야 하는 칸만 AI에게 받습니다. `order`, `code`, `kind`는 코드로 채웁니다.

In [ ]:
SOURCE_TYPES = ["read", "action", "input", "click"]

STEP_SCHEMA = {
    "type": "object",
    "properties": {
        "steps": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "title": {"type": "string", "description": "짧은 단계 제목"},
                    "instruction": {"type": "string", "description": "한 단계에 행동 하나. 짧은 문장."},
                    "expected": {"type": "string", "description": "성공하면 눈에 보이는 구체적 결과"},
                    "recovery": {"type": "string", "description": "실패했을 때 바로 할 수 있는 복구 방법"},
                    "sourceType": {"type": "string", "enum": SOURCE_TYPES},
                },
                "required": ["title", "instruction", "expected", "recovery", "sourceType"],
            },
        }
    },
    "required": ["steps"],
}

PROMPT = f"""
당신은 한국어 튜토리얼 편집자입니다. 주제: "{TOPIC}"
규칙:
- 단계 {STEP_COUNT}개를 만듭니다. 한 단계 = 행동 하나.
- 쉬운 한국어(ISO 24495-1 원칙): 짧은 문장, 결론 먼저, 전문용어는 처음 나올 때 풀어 씀.
- 메뉴 이름, 명령어, 고유명사는 원래 표기 그대로 씁니다.
- 확실하지 않은 메뉴 이름이나 명령은 지어내지 말고 끝에 "(확인 필요)"를 붙입니다.
- expected에는 성공했을 때 화면에서 보이는 것을 구체적으로 씁니다.
- recovery에는 실패했을 때 할 일을 씁니다.
- sourceType은 read(읽고 확인), action(실행), input(입력), click(누르기) 중 하나입니다.
"""

In [ ]:
# 5) 초안 생성
import json

response = client.models.generate_content(
    model=MODEL,
    contents=PROMPT,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_json_schema=STEP_SCHEMA,
    ),
)
draft = json.loads(response.text)
print(json.dumps(draft, ensure_ascii=False, indent=2))

## 6) 검증: 빠진 칸, 잘못된 값, 한 단계에 행동이 여러 개인지

In [ ]:
REQUIRED = ["title", "instruction", "expected", "recovery", "sourceType"]
problems = []
steps = draft.get("steps", [])
if not steps:
    problems.append("단계가 0개입니다.")
if steps and len(steps) != STEP_COUNT:
    problems.append(f"단계 수가 {len(steps)}개입니다. 요청은 {STEP_COUNT}개입니다.")
for i, s in enumerate(steps, start=1):
    for f in REQUIRED:
        if not str(s.get(f, "")).strip():
            problems.append(f"{i}단계: '{f}' 칸이 비어 있습니다.")
    if s.get("sourceType") not in SOURCE_TYPES:
        problems.append(f"{i}단계: sourceType '{s.get('sourceType')}' 은 허용 값이 아닙니다.")
    ins = str(s.get("instruction", ""))
    if " 그리고 " in ins or " 그다음 " in ins or len(ins) > 160:
        problems.append(f"{i}단계: 행동이 여러 개이거나 너무 깁니다. 나누는 것을 검토하세요.")
    if "확인 필요" in " ".join(str(v) for v in s.values()):
        problems.append(f"{i}단계: '(확인 필요)' 표시가 있습니다. 공식 문서로 확인하세요.")

print("문제 없음" if not problems else "\n".join(problems))

## 7) CSV 만들기
`steps` 탭에 붙여 넣을 형식입니다. **붙여 넣기 전에 사람이 검토하세요.**

In [ ]:
import pandas as pd

rows = []
for i, s in enumerate(steps, start=1):
    rows.append({
        "tutorial_id": TUTORIAL_ID,
        "order": i,
        "code": f"{CODE_PREFIX}.{i}",
        "title": s.get("title", f"{CODE_PREFIX}.{i}"),
        "instruction": s.get("instruction", ""),
        "expected": s.get("expected", ""),
        "recovery": s.get("recovery", ""),
        "kind": "manual",
        "sourceType": s.get("sourceType", ""),
    })
out = pd.DataFrame(rows, columns=["tutorial_id", "order", "code", "title", "instruction",
                                  "expected", "recovery", "kind", "sourceType"])
fname = f"steps_draft_{CODE_PREFIX}.csv"
out.to_csv(fname, index=False, encoding="utf-8-sig")
print("저장:", fname)
out

In [ ]:
# 8) 내 컴퓨터로 내려받기
from google.colab import files
files.download(fname)

---
**게시 전 확인표 (사람이 직접)**
- [ ] 단계마다 행동이 하나인가?
- [ ] 메뉴 이름과 명령을 공식 문서와 대조했는가?
- [ ] "(확인 필요)" 표시를 모두 해결했는가?
- [ ] expected가 눈으로 확인할 수 있는 결과인가?
- [ ] 키나 개인정보가 본문에 들어가지 않았는가?

모두 확인한 뒤 관리자 웹 앱에서 게시를 요청하세요.